In [7]:
import awswrangler as wr
import pyarrow as pa
import pandas as pd

In [8]:


# partitions example platform=Facebook/country=DE/year=2024/month=10/day=1/

schema = pa.schema([
    ("provider_data.rawlog_id", pa.string()),
    ("provider_data.user.id", pa.string()),
    ("provider_data.user.country", pa.string()),
    ("provider_data.waist", pa.string()),
    ("advertisement.details.advert_id", pa.string()),
    ("advertiser.details.advertiser_id", pa.string()),
    ("advertiser.details.advertiser_real_name", pa.string()),
    ("advertiser.details.advertiser_internal_name", pa.string()),
    ("advertiser.details.advertiser_website", pa.string()),
    ("targeting_categories", pa.string()),
    ("platform", pa.string()),
    ("day", pa.string()),
    ("month", pa.string()),
    ("year", pa.string()),
    ("country", pa.string()),
])


In [9]:

# This dataset only starts from 2024-08-29
df = pd.read_parquet(
  path='/Volumes/projects/whotargetsme/data/impressions-2025-04-10',
  filters=[
      [("country", "=", "DE"), ("year", "=", '2024')],
      [("country", "=", "DE"), ("year", "=", '2025')]
  ],
  schema=schema
)

In [10]:
# get monthly count for user
df_user_monthly = df.groupby(['provider_data.user.id', 'month']).size().reset_index(name='count')

# set minimum count threshold
df_user_monthly = df_user_monthly[df_user_monthly['count'] > 20]

# remove any rows where 'provider_data.user.id' group by is only 1 user i.e. missing data for a month
df_user_monthly = df_user_monthly.groupby('provider_data.user.id').filter(lambda x: len(x) > 2)

df_user_monthly

,provider_data.user.id,month,count
0,100021,1,104
1,100021,10,93
2,100021,11,86
3,100021,12,77
4,100021,2,68
...,...,...,...
15929,99304,2,21
15930,99304,3,45
15934,99462,1,140
15936,99462,11,25


In [11]:
# get unique `provider_data.user.id` from df_user_monthly
unique_user_ids = df_user_monthly['provider_data.user.id'].unique()

# cut list to 50 random ids
unique_user_ids = unique_user_ids[:50]

# filter df based on unique_user_ids
df_filtered = df[df['provider_data.user.id'].isin(unique_user_ids)]

df_filtered


,provider_data.rawlog_id,provider_data.user.id,provider_data.user.country,provider_data.waist,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,advertiser.details.advertiser_website,targeting_categories,platform,day,month,year,country
302,155770693,104652,DE,None,120210077035870669,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE
303,155770692,104652,DE,None,120211914952200338,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE
498,155768065,104652,DE,None,120209444818200741,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE
499,155768064,104652,DE,None,120213156686390494,None,None,None,None,"[{'name': 'INTERESTS', 'type': 'user_interest'...",Facebook,1,10,2024,DE
570,155767135,121411,DE,None,23853742225450785,None,None,None,None,"[{'name': 'CUSTOM_AUDIENCES_ENGAGEMENT_PAGE', ...",Facebook,1,10,2024,DE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4020229,162803159,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR06092760221497688065,CTW株式会社,CTW株式会社,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE
4020266,162803043,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,None,DISNEY STREAMING SERVICES LLC,DISNEY STREAMING SERVICES LLC,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE
4020267,162803042,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR05409422111431196673,attract mode GmbH,attract mode GmbH,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE
4020268,162803041,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR10942234166510485505,BlueVision Interactive Limited,BlueVision Interactive Limited,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE


In [12]:
# we'll import old postimpressions and merge that in, to get any missing waist data

df_old = pd.read_parquet(
  path='/Volumes/projects/whotargetsme/data/postimpressions-2025-04-10',
  filters=[
      [("country", "=", "DE"), ("year", "=", 2024)],
      [("country", "=", "DE"), ("year", "=", 2025)]
  ],
  columns=['waist_targeting_data', 'wtm_user_id', 'wtm_ref_rawlog_id']
)

df_old

,waist_targeting_data,wtm_user_id,wtm_ref_rawlog_id
0,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",95075,148032878
1,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",95075,148032879
2,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",86602,148032888
3,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",133714,148032897
4,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",95075,148032904
...,...,...,...
3811568,<NA>,70309,164648241
3811569,{'AGE_GENDER': {'__typename': 'WAISTUIAgeGende...,148761,164648264
3811570,{'AGE_GENDER': {'__typename': 'WAISTUIAgeGende...,93249,164648301
3811571,{'CUSTOM_AUDIENCES_ENGAGEMENT_PAGE': {'__typen...,93249,164648326


In [13]:
# merge on provider_data.rawlog_id, left join
df_merged = pd.merge(df_filtered, df_old, left_on='provider_data.rawlog_id', right_on='wtm_ref_rawlog_id', how='left', suffixes=('', '_remove') )

df_merged.drop([i for i in df_merged.columns if 'remove' in i],
               axis=1, inplace=True)

df_merged

,provider_data.rawlog_id,provider_data.user.id,provider_data.user.country,provider_data.waist,advertisement.details.advert_id,advertiser.details.advertiser_id,advertiser.details.advertiser_real_name,advertiser.details.advertiser_internal_name,advertiser.details.advertiser_website,targeting_categories,platform,day,month,year,country,waist_targeting_data,wtm_user_id,wtm_ref_rawlog_id
0,155770693,104652,DE,None,120210077035870669,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",104652.0,155770693
1,155770692,104652,DE,None,120211914952200338,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",104652.0,155770692
2,155768065,104652,DE,None,120209444818200741,None,None,None,None,"[{'name': 'AGE_GENDER', 'type': 'user_characte...",Facebook,1,10,2024,DE,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",104652.0,155768065
3,155768064,104652,DE,None,120213156686390494,None,None,None,None,"[{'name': 'INTERESTS', 'type': 'user_interest'...",Facebook,1,10,2024,DE,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",104652.0,155768064
4,155767135,121411,DE,None,23853742225450785,None,None,None,None,"[{'name': 'CUSTOM_AUDIENCES_ENGAGEMENT_PAGE', ...",Facebook,1,10,2024,DE,"{'ACTIONABLE_INSIGHTS': None, 'AGE_GENDER': {'...",121411.0,155767135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60483,162803159,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR06092760221497688065,CTW株式会社,CTW株式会社,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE,<NA>,NaN,<NA>
60484,162803043,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,None,DISNEY STREAMING SERVICES LLC,DISNEY STREAMING SERVICES LLC,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE,<NA>,NaN,<NA>
60485,162803042,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR05409422111431196673,attract mode GmbH,attract mode GmbH,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE,<NA>,NaN,<NA>
60486,162803041,100820,DE,"<div role=""region"" aria-labelledby=""KIBZ7b""><d...",,AR10942234166510485505,BlueVision Interactive Limited,BlueVision Interactive Limited,https://www.googleadservices.com/pagead/aclk?s...,"[{'name': '', 'type': 'other', 'value': 'The t...",Youtube,9,3,2025,DE,<NA>,NaN,<NA>


In [14]:
# write to local csv
df_merged.to_csv('study-2/waist_impressions_DE_random_50_users_v3.csv')